# Inference on Modal

## Importing the libraries

In [9]:
import os
import modal
import locale
from modal import Image
from dotenv import load_dotenv

## Loading Modal's Token and Secret

In [ ]:
load_dotenv(override = True)
token_id = os.getenv("MODAL_TOKEN_ID")
token_secret = os.getenv("MODAL_TOKEN_SECRET")

if token_id and token_secret:
    print("API Keys Found!")
else:
    print("API Keys not Found!")

API Key Found!


## Creating Modal App

In [11]:
app = modal.App("inference-on-modal")
image = Image.debian_slim().pip_install("requests")

## Creating Modal Function

In [12]:
@app.function(image = image)
def remote():

    import requests

    response = requests.get("https://ipinfo.io/json")
    data = response.json()

    city, region, country = data["city"], data["region"], data["country"]

    return f"Hello form {city}, {region}, {country}"

## Running the Modal Function Locally

In [15]:
with app.run():
    response = remote.local()

In [16]:
print(response)

Hello form Lahore, Punjab, PK


## Running the Modal Function Remotely

In [17]:
with app.run():
    response = remote.remote()

In [18]:
print(response)

Hello form Columbus, Ohio, US


## Deploying Model as Class

In [ ]:
!uv run modal deploy -m llama_scribe

In [25]:
LlamaScribe = modal.Cls.from_name("llama", "LlamaScribe")
llama_scribe = LlamaScribe()

In [26]:
prompt = "Extract the relevant fields from this OCR text, ignoring any unrelated text or artifacts:\n\"CITY HEALTH CLINIC\\nBILL NO: HC-48291\\nDATE: 08/MAR/2026\\nPATIENT: SARAH LIM\\nCONSULTATION $55.00\\nMEDICATION $25.00\\nGST (9%): $7.20\\nTOTAL: $87.20\\nREMARKS: FOLLOW-UP VISIT SUBSIDY APPLIED.\\nPAGE 1 OF 1\"\n"

In [ ]:
response = llama_scribe.llama_scribe.remote(prompt)
print(response[response.find("{"):])

{
  "doc_type": "medical_bill",
  "schema": {
    "Date_of_Visit": "08/MAR/2026",
    "Date_of_Visit_confidence_score": 98,
    "Amt": "87.20",
    "Amt_confidence_score": 99,
    "GST": "7.20",
    "GST_confidence_score": 95,
    "Remarks": "FOLLOW-UP VISIT SUBSIDY APPLIED.",
    "Remarks_confidence_score": 86
  }
}
